# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [ ]:
# Setup
# pip install -r requirements.txt
import os, json
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

# NOTE: Using Ollama as Gemini API quota was exhausted (429 RESOURCE_EXHAUSTED)
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama" # required but ignored
)
MODEL = "llama3.2:3b"
LABELS = ["billing", "bug", "feature_request", "other"]

## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [ ]:
# 1. Make a working call: print response and token usage
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise support assistant"},
        {"role": "user", "content": "Tell me a joke about AI."}
    ]
)
print(f"Response: {response.choices[0].message.content}")
print(f"Usage: {response.usage}")

# 2. Temperature loop
prompt = f"Classify this support ticket into one of these labels: {', '.join(LABELS)}. Ticket: 'I was charged twice'. Just the label."

for temp in [0.1, 1.0]:
    print(f"\nTemperature: {temp}")
    for i in range(3):
        res = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a concise support assistant"},
                {"role": "user", "content": prompt}
            ],
            temperature=temp
        )
        print(f"Run {i+1}: {res.choices[0].message.content.strip()}")

**What changed, and why?**

At **temperature 0.1**, the model is very deterministic, often producing the exact same output across multiple runs. This is because it greedily selects the highest probability tokens. At **temperature 1.0**, the output becomes more stochastic and creative. While for a simple classification task the label might remain the same, the model might add extra punctuation, change casing, or vary its phrasing more often as it samples from a wider probability distribution of tokens.

## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [ ]:
with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

def classify(text, style):
    system_message = "You are a support ticket classifier. Only output the label (billing, bug, feature_request, or other)."
    
    if style == "zero_shot":
        user_message = f"Classify this ticket: '{text}'"
    elif style == "few_shot":
        user_message = (
            "Ticket: 'My app keeps crashing' -> Label: bug\n"
            "Ticket: 'Can I get a refund?' -> Label: billing\n"
            "Ticket: 'I love the new update' -> Label: other\n"
            f"Ticket: '{text}' -> Label:"
        )
    elif style == "cot":
        user_message = f"Classify this ticket: '{text}'. First, briefly reason about the category, then provide the label on a new line starting with 'Label: '."
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=0.1
    )
    
    content = response.choices[0].message.content.strip()
    
    if style == "cot":
        # Extract label from COT output
        for line in content.split("\n"):
            if "Label:" in line:
                return line.split("Label:")[1].strip().lower()
    
    # Clean up the output to match labels
    for label in LABELS:
        if label in content.lower():
            return label
    return "other"

results = []
for t in tickets:
    results.append({
        "Ticket": t["text"],
        "Zero-Shot": classify(t["text"], "zero_shot"),
        "Few-Shot": classify(t["text"], "few_shot"),
        "CoT": classify(t["text"], "cot")
    })

df = pd.DataFrame(results)
print(df.to_markdown(index=False))

## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [ ]:
def classify_structured(text):
    prompt = f"""Classify this support ticket: '{text}'
    Return ONLY a JSON object with exactly these keys:
    - label: one of billing, bug, feature_request, other
    - confidence: a float between 0 and 1
    - reason: a short string explaining the choice
    """
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a support assistant that only outputs JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            response_format={"type": "json_object"} # Ollama supports this
        )
        
        raw_json = response.choices[0].message.content
        data = json.loads(raw_json)
        
        # Validation
        if data.get("label") not in LABELS:
            data["label"] = "other"
        if not (0 <= data.get("confidence", 0) <= 1):
            data["confidence"] = 0.0
            
        return data
    except Exception as e:
        return {"label": "other", "confidence": 0.0, "reason": f"Error parsing: {str(e)}"}

# Test Task 3
print(json.dumps(classify_structured(tickets[0]["text"]), indent=2))
print(json.dumps(classify_structured(tickets[1]["text"]), indent=2))

**Local vs hosted: did the small local model produce valid JSON as reliably?**

Since I am running the entire lab using the local Ollama model due to Gemini quota limits, I observed that `llama3.2:3b` is remarkably reliable at producing JSON when using the `response_format={"type": "json_object"}` parameter. However, without that hint, small models often struggle with trailing commas or markdown code blocks. Compared to hosted models like Gemini or GPT-4o, the local model is faster for simple tasks but may fail more often on complex validation or reasoning within the JSON fields.